# Eye tracking + FTAR features experiments

This notebook mirrors the text2text experiment workflow, but uses FTAR feature sets from:
`C:\Users\LEGION\Projects\CB_exepriment\ftar_features`

Targets/splits covered:
- `match_mismatch`: binary + multiclass
- `match_mismatch_general`: binary + multiclass

In [1]:
from __future__ import annotations

import os
import json
from pathlib import Path
from typing import Tuple

import numpy as np
import pandas as pd
from umap import UMAP

from experiment_code.read_data import get_data_for_split
from experiment_code.run_experiments import _build_groups_mm_for_mm
from experiment_code.optuna_code import run_and_log, fit_best_and_test

ROOT = Path(r"C:\Users\LEGION\data\СB")
os.chdir(ROOT)
print("cwd:", Path.cwd())

cwd: C:\Users\LEGION\data\СB


In [2]:
TARGETS = ["match_mismatch", "match_mismatch_general"]
SPLITS = ["binary", "multiclass"]

USE_EARLY_STOPPING = True
REFIT_CV = False
REFIT_TEST = True
USE_MIN = False

CV = 4
N_TRIALS_XGB = 100
N_TRIALS_CB = 50

PROJECT_DIR = Path(r"C:\Users\LEGION\Projects\CB_exepriment\dataset_v2")
OUT_DIR = PROJECT_DIR / "optuna_results_eye_ftar"
OUT_DIR.mkdir(parents=True, exist_ok=True)

FI_DIR = OUT_DIR / "feature_importances_eye_ftar"
FI_DIR.mkdir(parents=True, exist_ok=True)
REFIT_TOP_N = 30

UMAP_TARGET_SETS = {"freq_bands", "stat", "corr", "cov_freq"}
UMAP_N_COMPONENTS = 100
UMAP_N_NEIGHBORS = 30
UMAP_MIN_DIST = 0.0
UMAP_METRIC = "cosine"
UMAP_RANDOM_STATE = 1717

CLEAN_TRAIN_COLUMNS = False

In [3]:
def _prefix_cols(df: pd.DataFrame, prefix: str) -> pd.DataFrame:
    return df.add_prefix(prefix)


def _load_ftar_parquet(path: Path) -> pd.DataFrame:
    df = pd.read_parquet(path)
    if "pid_rn" not in df.columns:
        raise KeyError(f"'pid_rn' column not found in {path}")
    df = df.set_index("pid_rn")
    df.index = df.index.astype(str)
    return df


def _apply_umap_train_test(
    text_train: pd.DataFrame,
    text_test: pd.DataFrame,
    *,
    text_set_name: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    tr = text_train.apply(pd.to_numeric, errors="coerce")
    te = text_test.apply(pd.to_numeric, errors="coerce")

    train_means = tr.mean(numeric_only=True)
    tr = tr.fillna(train_means).fillna(0.0)
    te = te.fillna(train_means).fillna(0.0)

    n_components = int(min(UMAP_N_COMPONENTS, max(2, tr.shape[1])))
    reducer = UMAP(
        n_components=n_components,
        n_neighbors=UMAP_N_NEIGHBORS,
        min_dist=UMAP_MIN_DIST,
        metric=UMAP_METRIC,
        random_state=UMAP_RANDOM_STATE,
    )

    print(
        f"[{text_set_name}] UMAP compression: "
        f"train/test features {tr.shape[1]} -> {n_components}"
    )

    z_train = reducer.fit_transform(tr)
    z_test = reducer.transform(te)

    cols = [f"{text_set_name}__umap_{i:03d}" for i in range(n_components)]
    tr_umap = pd.DataFrame(z_train, index=text_train.index, columns=cols)
    te_umap = pd.DataFrame(z_test, index=text_test.index, columns=cols)
    return tr_umap, te_umap


def _prepare_joined_X(
    eye_train: pd.DataFrame,
    eye_test: pd.DataFrame,
    text_all: pd.DataFrame,
    text_set_name: str,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    text_train = text_all.reindex(eye_train.index)
    text_test = text_all.reindex(eye_test.index)

    missing_train = int(text_train.isna().all(axis=1).sum())
    missing_test = int(text_test.isna().all(axis=1).sum())
    print(f"[{text_set_name}] rows missing after index match -> train: {missing_train}, test: {missing_test}")

    if text_set_name in UMAP_TARGET_SETS:
        text_train_model, text_test_model = _apply_umap_train_test(
            text_train,
            text_test,
            text_set_name=text_set_name,
        )
    else:
        text_train_model = _prefix_cols(text_train, f"{text_set_name}__")
        text_test_model = _prefix_cols(text_test, f"{text_set_name}__")

    X_train = pd.concat([
        _prefix_cols(eye_train, "eye__"),
        text_train_model,
    ], axis=1)
    X_test = pd.concat([
        _prefix_cols(eye_test, "eye__"),
        text_test_model,
    ], axis=1)

    if CLEAN_TRAIN_COLUMNS:
        keep_cols = X_train.columns[X_train.notna().all(axis=0)]
        X_train = X_train[keep_cols]
        X_test = X_test.reindex(columns=keep_cols)

    return X_train, X_test


def _build_no_neutral_mask(stim_df: pd.DataFrame, target: str, split: str) -> pd.Series:
    mask = pd.Series(True, index=stim_df.index)

    if "valence" in stim_df.columns:
        valence = pd.to_numeric(stim_df["valence"], errors="coerce")
        mask &= valence != 2

    if split == "binary":
        if target == "match_mismatch_general" and "exp_general_multi" in stim_df.columns:
            exp_general_multi = pd.to_numeric(stim_df["exp_general_multi"], errors="coerce")
            mask &= exp_general_multi != 2
        elif "IAT_results2" in stim_df.columns:
            iat = stim_df["IAT_results2"].astype(str).str.strip().str.lower()
            mask &= iat != "neutral"

    return mask.fillna(False)


def _load_eye_for_task(target: str, split: str, no_neutral: bool = False):
    df, tgt, _ = get_data_for_split(X_name="screen", target=target, split=split)

    eye_train = df["all_features"]["X_train"].copy()
    eye_test = df["all_features"]["X_test"].copy()
    stim_train = df["stimuli_features"]["X_train"].copy()
    stim_test = df["stimuli_features"]["X_test"].copy()

    y_train = tgt["cb"]["y_train"].copy()
    y_test = tgt["cb"]["y_test"].copy()

    if no_neutral:
        mask_train = _build_no_neutral_mask(stim_train, target=target, split=split)
        mask_test = _build_no_neutral_mask(stim_test, target=target, split=split)

        eye_train = eye_train.loc[mask_train]
        eye_test = eye_test.loc[mask_test]
        y_train = y_train.loc[mask_train]
        y_test = y_test.loc[mask_test]
        stim_train = stim_train.loc[mask_train]

    groups = np.array([str(i).split("_")[0] for i in eye_train.index])

    if target in ("match_mismatch", "match_mismatch_general"):
        groups_mm = _build_groups_mm_for_mm(X_train_index=eye_train.index, stim_train_df=stim_train)
    else:
        groups_mm = None

    problem = "binary" if split == "binary" else "multiclass"
    return eye_train, eye_test, y_train, y_test, groups, groups_mm, problem


def run_ftar_set(
    text_set_name: str,
    text_all: pd.DataFrame,
    *,
    do_refit: bool = True,
    top_n_features: int = REFIT_TOP_N,
):
    if "pid_rn" in text_all.columns:
        text_all = text_all.set_index("pid_rn")
    text_all = text_all.copy()
    text_all.index = text_all.index.astype(str)

    if "key" in text_all.columns:
        text_all = text_all.drop(columns=["key"])

    print(f"[{text_set_name}] ftar feature count after cleanup: {int(text_all.shape[1])}")

    rows = []

    for target in TARGETS:
        for split in SPLITS:
            for no_neutral in [False, True]:
                problem = "binary" if split == "binary" else "multiclass"
                neutral_suffix = "__no_neutral" if no_neutral else ""
                train_features_name = (
                    f"exp__X_name=screen+{text_set_name}"
                    f"__target={target}"
                    f"__problem={problem}"
                    f"__feat=all_features+{text_set_name}"
                    f"__ES__refitTEST"
                    f"{neutral_suffix}"
                )
                out_path = OUT_DIR / f"{train_features_name}.json"

                if out_path.exists():
                    print("\n" + "=" * 80)
                    print(f"Skipping existing experiment: {train_features_name}")
                    with open(out_path, encoding="utf-8") as f:
                        results = json.load(f)
                    rows.append({
                        "set": text_set_name,
                        "target": target,
                        "split": split,
                        "no_neutral": no_neutral,
                        "xgb_test": float(results["xgb"]["test_metrics"]["primary"]),
                        "catboost_test": float(results["catboost"]["test_metrics"]["primary"]),
                        "file": str(out_path),
                    })
                    continue

                eye_train, eye_test, y_train, y_test, groups, groups_mm, problem = _load_eye_for_task(
                    target,
                    split,
                    no_neutral=no_neutral,
                )
                X_train, X_test = _prepare_joined_X(eye_train, eye_test, text_all, text_set_name)

                print("\n" + "=" * 80)
                print(f"Running: {train_features_name}")
                print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

                results = run_and_log(
                    train_features=train_features_name,
                    problem=problem,
                    X_train=X_train,
                    X_test=X_test,
                    y_train=y_train,
                    y_test=y_test,
                    strat_train=y_train,
                    groups=groups,
                    groups_mm=groups_mm,
                    use_early_stopping=USE_EARLY_STOPPING,
                    refit_cv=REFIT_CV,
                    refit_test=REFIT_TEST,
                    n_trials_xgb=N_TRIALS_XGB,
                    n_trials_cb=N_TRIALS_CB,
                    cv=CV,
                    gpu=False,
                    use_min=USE_MIN,
                    cat_cols=None,
                    target_names=None,
                    out_path=out_path,
                )

                if do_refit:
                    print("\n" + "-" * 80)
                    print("Refit best iteration + metrics + top features")
                    for model_name in ["xgb", "catboost"]:
                        best_params = results[model_name].get("suggested_params", {})
                        if not best_params:
                            print(f"\n=== Skip refit {model_name.upper()} (no suggested params) ===")
                            continue

                        print(f"\n=== Refit {model_name.upper()} with saved best params ===")
                        out_refit = fit_best_and_test(
                            model_name=model_name,
                            best_params=best_params,
                            problem=problem,
                            X_train=X_train,
                            X_test=X_test,
                            y_train=y_train,
                            y_test=y_test,
                            strat_train=y_train,
                            groups=groups,
                            groups_mm=groups_mm,
                            use_early_stopping=True,
                            early_stopping_rounds=100,
                            target_names=None,
                            cat_cols=None,
                            gpu=False,
                            refit_test=True,
                        )

                        model = out_refit["model"]
                        if model_name == "xgb":
                            importances = model.feature_importances_
                        else:
                            importances = model.get_feature_importance()

                        fi = pd.DataFrame({
                            "feature": X_train.columns,
                            "importance": importances,
                        }).sort_values("importance", ascending=False)

                        fi_path = FI_DIR / f"{train_features_name}__{model_name}.csv"
                        fi.to_csv(fi_path, index=False)
                        print(f"Saved feature importances: {fi_path}")
                        print(f"Top {top_n_features} features for {model_name.upper()}:")
                        display(fi.head(top_n_features))

                rows.append({
                    "set": text_set_name,
                    "target": target,
                    "split": split,
                    "no_neutral": no_neutral,
                    "xgb_test": float(results["xgb"]["test_metrics"]["primary"]),
                    "catboost_test": float(results["catboost"]["test_metrics"]["primary"]),
                    "file": str(out_path),
                })

    summary = pd.DataFrame(rows)
    display(summary)
    return summary

In [4]:
FTAR_DIR = Path(r"C:\Users\LEGION\Projects\CB_exepriment\ftar_features")
FTAR_FILES = {
    "corr": FTAR_DIR / "corr.parquet",
    "cov_freq": FTAR_DIR / "cov_freq.parquet",
    "envelope": FTAR_DIR / "envelope.parquet",
    "freq_bands": FTAR_DIR / "freq_bands.parquet",
    "PID": FTAR_DIR / "PID.parquet",
    "stat": FTAR_DIR / "stat.parquet",
}

print("ftar dir:", FTAR_DIR)
for k, p in FTAR_FILES.items():
    print(f"{k:10s} -> {p} | exists={p.exists()}")

ftar dir: C:\Users\LEGION\Projects\CB_exepriment\ftar_features
corr       -> C:\Users\LEGION\Projects\CB_exepriment\ftar_features\corr.parquet | exists=True
cov_freq   -> C:\Users\LEGION\Projects\CB_exepriment\ftar_features\cov_freq.parquet | exists=True
envelope   -> C:\Users\LEGION\Projects\CB_exepriment\ftar_features\envelope.parquet | exists=True
freq_bands -> C:\Users\LEGION\Projects\CB_exepriment\ftar_features\freq_bands.parquet | exists=True
PID        -> C:\Users\LEGION\Projects\CB_exepriment\ftar_features\PID.parquet | exists=True
stat       -> C:\Users\LEGION\Projects\CB_exepriment\ftar_features\stat.parquet | exists=True


In [5]:
text_corr = _load_ftar_parquet(FTAR_FILES["corr"])
print("corr shape:", text_corr.shape)
summary_corr = run_ftar_set("corr", text_corr)
summary_corr

corr shape: (4442, 1831)
[corr] ftar feature count after cleanup: 1830

Skipping existing experiment: exp__X_name=screen+corr__target=match_mismatch__problem=binary__feat=all_features+corr__ES__refitTEST
[corr] rows missing after index match -> train: 1658, test: 160
[corr] UMAP compression: train/test features 1830 -> 100


c:\Users\LEGION\miniconda3\envs\table_methods\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
[I 2026-04-15 09:25:58,718] A new study created in memory with name: binary_xgb



Running: exp__X_name=screen+corr__target=match_mismatch__problem=binary__feat=all_features+corr__ES__refitTEST__no_neutral
X_train: (3016, 570), X_test: (336, 570)


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-04-15 09:26:03,076] Trial 0 finished with value: 0.40517617025578806 and parameters: {'learning_rate': 0.04534239315199742, 'max_depth': 4, 'min_child_weight': 3.6133378418934727, 'subsample': 0.7172121707956832, 'colsample_bytree': 0.7642269046455532, 'gamma': 0.2930268354654828, 'reg_alpha': 0.0005749550436838274, 'reg_lambda': 2.9181778067053106}. Best is trial 0 with value: 0.40517617025578806.
[I 2026-04-15 09:26:08,305] Trial 1 finished with value: 0.40517617025578806 and parameters: {'learning_rate': 0.1707594054749505, 'max_depth': 8, 'min_child_weight': 1.8407677988385938, 'subsample': 0.9514458866935752, 'colsample_bytree': 0.7320289895353271, 'gamma': 0.030927525032881986, 'reg_alpha': 2.3384483013779853, 'reg_lambda': 0.0007548524119611296}. Best is trial 0 with value: 0.40517617025578806.
[I 2026-04-15 09:26:13,082] Trial 2 finished with value: 0.40517617025578806 and parameters: {'learning_rate': 0.0333616681035175, 'max_depth': 5, 'min_child_weight': 1.3687627858

...

  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-04-15 10:41:14,109] Trial 0 finished with value: 0.6344958901424369 and parameters: {'bootstrap_type': 'Bayesian', 'learning_rate': 0.07226675683786944, 'depth': 4, 'l2_leaf_reg': 1.8990530558141054, 'random_strength': 0.1465134177327414, 'grow_policy': 'SymmetricTree', 'leaf_estimation_method': 'Gradient', 'leaf_estimation_iterations': 5, 'bagging_temperature': 4.973275478387985}. Best is trial 0 with value: 0.6344958901424369.
[I 2026-04-15 10:41:25,995] Trial 1 finished with value: 0.6304976964539113 and parameters: {'bootstrap_type': 'Bernoulli', 'learning_rate': 0.025573683975272207, 'depth': 4, 'l2_leaf_reg': 13.70317538683857, 'random_strength': 0.1755724094154989, 'grow_policy': 'SymmetricTree', 'leaf_estimation_method': 'Gradient', 'leaf_estimation_iterations': 1, 'subsample': 0.6972589059086052}. Best is trial 0 with value: 0.6344958901424369.
[I 2026-04-15 10:41:40,485] Trial 2 finished with value: 0.6355436834734819 and parameters: {'bootstrap_type': 'Bernoulli', 'l

,feature,importance
53,eye__sac_length_AOI_aggregated_Pos_min,10.029127
56,eye__sac_length_AOI_aggregated_Pos_median,9.779052
60,eye__sac_speed_AOI_aggregated_Pos_min,9.540814
206,eye__fix_duration_AOI_FD_Pos1_min,4.480646
276,eye__fix_duration_AOI_aggregated_Pos_min,3.895406
63,eye__sac_speed_AOI_aggregated_Pos_median,3.728141
54,eye__sac_length_AOI_aggregated_Pos_max,2.437045
38,eye__sac_speed_AOI_aggregated_Neg_mean,1.801476
541,corr__umap_071,1.280923
263,eye__fix_duration_AOI_aggregated_Neg_sum,1.255980


,set,target,split,no_neutral,xgb_test,catboost_test,file
0,corr,match_mismatch,binary,False,0.398399,0.414870,C:\Users\LEGION\Projects\CB_exepriment\dataset...
1,corr,match_mismatch,binary,True,0.403197,0.443771,C:\Users\LEGION\Projects\CB_exepriment\dataset...
2,corr,match_mismatch,multiclass,False,0.596315,0.608868,C:\Users\LEGION\Projects\CB_exepriment\dataset...
3,corr,match_mismatch,multiclass,True,NaN,0.541407,C:\Users\LEGION\Projects\CB_exepriment\dataset...
4,corr,match_mismatch_general,binary,False,0.426946,0.529915,C:\Users\LEGION\Projects\CB_exepriment\dataset...
5,corr,match_mismatch_general,binary,True,0.440106,0.486745,C:\Users\LEGION\Projects\CB_exepriment\dataset...
6,corr,match_mismatch_general,multiclass,False,0.799472,0.808972,C:\Users\LEGION\Projects\CB_exepriment\dataset...
7,corr,match_mismatch_general,multiclass,True,NaN,0.773737,C:\Users\LEGION\Projects\CB_exepriment\dataset...


,set,target,split,no_neutral,xgb_test,catboost_test,file
0,corr,match_mismatch,binary,False,0.398399,0.414870,C:\Users\LEGION\Projects\CB_exepriment\dataset...
1,corr,match_mismatch,binary,True,0.403197,0.443771,C:\Users\LEGION\Projects\CB_exepriment\dataset...
2,corr,match_mismatch,multiclass,False,0.596315,0.608868,C:\Users\LEGION\Projects\CB_exepriment\dataset...
3,corr,match_mismatch,multiclass,True,NaN,0.541407,C:\Users\LEGION\Projects\CB_exepriment\dataset...
4,corr,match_mismatch_general,binary,False,0.426946,0.529915,C:\Users\LEGION\Projects\CB_exepriment\dataset...
5,corr,match_mismatch_general,binary,True,0.440106,0.486745,C:\Users\LEGION\Projects\CB_exepriment\dataset...
6,corr,match_mismatch_general,multiclass,False,0.799472,0.808972,C:\Users\LEGION\Projects\CB_exepriment\dataset...
7,corr,match_mismatch_general,multiclass,True,NaN,0.773737,C:\Users\LEGION\Projects\CB_exepriment\dataset...


In [6]:
text_cov_freq = _load_ftar_parquet(FTAR_FILES["cov_freq"])
print("cov_freq shape:", text_cov_freq.shape)
summary_cov_freq = run_ftar_set("cov_freq", text_cov_freq)
summary_cov_freq

cov_freq shape: (4442, 36296)
[cov_freq] ftar feature count after cleanup: 36295

Skipping existing experiment: exp__X_name=screen+cov_freq__target=match_mismatch__problem=binary__feat=all_features+cov_freq__ES__refitTEST
[cov_freq] rows missing after index match -> train: 1658, test: 160
[cov_freq] UMAP compression: train/test features 36295 -> 100


c:\Users\LEGION\miniconda3\envs\table_methods\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
[I 2026-04-15 11:00:52,723] A new study created in memory with name: binary_xgb



Running: exp__X_name=screen+cov_freq__target=match_mismatch__problem=binary__feat=all_features+cov_freq__ES__refitTEST__no_neutral
X_train: (3016, 570), X_test: (336, 570)


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-04-15 11:00:56,443] Trial 0 finished with value: 0.40517617025578806 and parameters: {'learning_rate': 0.04534239315199742, 'max_depth': 4, 'min_child_weight': 3.6133378418934727, 'subsample': 0.7172121707956832, 'colsample_bytree': 0.7642269046455532, 'gamma': 0.2930268354654828, 'reg_alpha': 0.0005749550436838274, 'reg_lambda': 2.9181778067053106}. Best is trial 0 with value: 0.40517617025578806.
[I 2026-04-15 11:01:01,176] Trial 1 finished with value: 0.40517617025578806 and parameters: {'learning_rate': 0.1707594054749505, 'max_depth': 8, 'min_child_weight': 1.8407677988385938, 'subsample': 0.9514458866935752, 'colsample_bytree': 0.7320289895353271, 'gamma': 0.030927525032881986, 'reg_alpha': 2.3384483013779853, 'reg_lambda': 0.0007548524119611296}. Best is trial 0 with value: 0.40517617025578806.
[I 2026-04-15 11:01:05,824] Trial 2 finished with value: 0.40401640806135186 and parameters: {'learning_rate': 0.0333616681035175, 'max_depth': 5, 'min_child_weight': 1.3687627858

...

  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-04-15 12:36:46,511] Trial 0 finished with value: 0.6400107572122057 and parameters: {'bootstrap_type': 'Bayesian', 'learning_rate': 0.07226675683786944, 'depth': 4, 'l2_leaf_reg': 1.8990530558141054, 'random_strength': 0.1465134177327414, 'grow_policy': 'SymmetricTree', 'leaf_estimation_method': 'Gradient', 'leaf_estimation_iterations': 5, 'bagging_temperature': 4.973275478387985}. Best is trial 0 with value: 0.6400107572122057.
[I 2026-04-15 12:36:56,831] Trial 1 finished with value: 0.6539717650356516 and parameters: {'bootstrap_type': 'Bernoulli', 'learning_rate': 0.025573683975272207, 'depth': 4, 'l2_leaf_reg': 13.70317538683857, 'random_strength': 0.1755724094154989, 'grow_policy': 'SymmetricTree', 'leaf_estimation_method': 'Gradient', 'leaf_estimation_iterations': 1, 'subsample': 0.6972589059086052}. Best is trial 1 with value: 0.6539717650356516.
[I 2026-04-15 12:37:11,550] Trial 2 finished with value: 0.643703099405881 and parameters: {'bootstrap_type': 'Bernoulli', 'le

,feature,importance
63,eye__sac_speed_AOI_aggregated_Pos_median,10.337758
60,eye__sac_speed_AOI_aggregated_Pos_min,8.754785
53,eye__sac_length_AOI_aggregated_Pos_min,8.539778
56,eye__sac_length_AOI_aggregated_Pos_median,7.335902
276,eye__fix_duration_AOI_aggregated_Pos_min,5.024279
277,eye__fix_duration_AOI_aggregated_Pos_max,3.741627
61,eye__sac_speed_AOI_aggregated_Pos_max,3.449336
278,eye__fix_duration_AOI_aggregated_Pos_mean,2.678137
55,eye__sac_length_AOI_aggregated_Pos_mean,2.384657
279,eye__fix_duration_AOI_aggregated_Pos_median,2.093244


,set,target,split,no_neutral,xgb_test,catboost_test,file
0,cov_freq,match_mismatch,binary,False,0.399740,0.403406,C:\Users\LEGION\Projects\CB_exepriment\dataset...
1,cov_freq,match_mismatch,binary,True,0.403197,0.505965,C:\Users\LEGION\Projects\CB_exepriment\dataset...
2,cov_freq,match_mismatch,multiclass,False,0.621595,0.628659,C:\Users\LEGION\Projects\CB_exepriment\dataset...
3,cov_freq,match_mismatch,multiclass,True,NaN,0.544607,C:\Users\LEGION\Projects\CB_exepriment\dataset...
4,cov_freq,match_mismatch_general,binary,False,0.398957,0.576895,C:\Users\LEGION\Projects\CB_exepriment\dataset...
5,cov_freq,match_mismatch_general,binary,True,0.410526,0.562286,C:\Users\LEGION\Projects\CB_exepriment\dataset...
6,cov_freq,match_mismatch_general,multiclass,False,0.802246,0.803497,C:\Users\LEGION\Projects\CB_exepriment\dataset...
7,cov_freq,match_mismatch_general,multiclass,True,NaN,0.772836,C:\Users\LEGION\Projects\CB_exepriment\dataset...


,set,target,split,no_neutral,xgb_test,catboost_test,file
0,cov_freq,match_mismatch,binary,False,0.399740,0.403406,C:\Users\LEGION\Projects\CB_exepriment\dataset...
1,cov_freq,match_mismatch,binary,True,0.403197,0.505965,C:\Users\LEGION\Projects\CB_exepriment\dataset...
2,cov_freq,match_mismatch,multiclass,False,0.621595,0.628659,C:\Users\LEGION\Projects\CB_exepriment\dataset...
3,cov_freq,match_mismatch,multiclass,True,NaN,0.544607,C:\Users\LEGION\Projects\CB_exepriment\dataset...
4,cov_freq,match_mismatch_general,binary,False,0.398957,0.576895,C:\Users\LEGION\Projects\CB_exepriment\dataset...
5,cov_freq,match_mismatch_general,binary,True,0.410526,0.562286,C:\Users\LEGION\Projects\CB_exepriment\dataset...
6,cov_freq,match_mismatch_general,multiclass,False,0.802246,0.803497,C:\Users\LEGION\Projects\CB_exepriment\dataset...
7,cov_freq,match_mismatch_general,multiclass,True,NaN,0.772836,C:\Users\LEGION\Projects\CB_exepriment\dataset...


In [7]:
text_envelope = _load_ftar_parquet(FTAR_FILES["envelope"])
print("envelope shape:", text_envelope.shape)
summary_envelope = run_ftar_set("envelope", text_envelope)
summary_envelope

envelope shape: (4442, 428)
[envelope] ftar feature count after cleanup: 427

Skipping existing experiment: exp__X_name=screen+envelope__target=match_mismatch__problem=binary__feat=all_features+envelope__ES__refitTEST


[I 2026-04-15 12:53:23,933] A new study created in memory with name: binary_xgb


[envelope] rows missing after index match -> train: 1658, test: 160

Running: exp__X_name=screen+envelope__target=match_mismatch__problem=binary__feat=all_features+envelope__ES__refitTEST__no_neutral
X_train: (3016, 897), X_test: (336, 897)


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-04-15 12:53:31,686] Trial 0 finished with value: 0.40517617025578806 and parameters: {'learning_rate': 0.04534239315199742, 'max_depth': 4, 'min_child_weight': 3.6133378418934727, 'subsample': 0.7172121707956832, 'colsample_bytree': 0.7642269046455532, 'gamma': 0.2930268354654828, 'reg_alpha': 0.0005749550436838274, 'reg_lambda': 2.9181778067053106}. Best is trial 0 with value: 0.40517617025578806.
[I 2026-04-15 12:53:41,067] Trial 1 finished with value: 0.40517617025578806 and parameters: {'learning_rate': 0.1707594054749505, 'max_depth': 8, 'min_child_weight': 1.8407677988385938, 'subsample': 0.9514458866935752, 'colsample_bytree': 0.7320289895353271, 'gamma': 0.030927525032881986, 'reg_alpha': 2.3384483013779853, 'reg_lambda': 0.0007548524119611296}. Best is trial 0 with value: 0.40517617025578806.
[I 2026-04-15 12:53:49,983] Trial 2 finished with value: 0.40517617025578806 and parameters: {'learning_rate': 0.0333616681035175, 'max_depth': 5, 'min_child_weight': 1.3687627858

...

  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-04-15 14:56:55,142] Trial 0 finished with value: 0.6352743426672984 and parameters: {'bootstrap_type': 'Bayesian', 'learning_rate': 0.07226675683786944, 'depth': 4, 'l2_leaf_reg': 1.8990530558141054, 'random_strength': 0.1465134177327414, 'grow_policy': 'SymmetricTree', 'leaf_estimation_method': 'Gradient', 'leaf_estimation_iterations': 5, 'bagging_temperature': 4.973275478387985}. Best is trial 0 with value: 0.6352743426672984.
[I 2026-04-15 14:57:10,739] Trial 1 finished with value: 0.6553038926072227 and parameters: {'bootstrap_type': 'Bernoulli', 'learning_rate': 0.025573683975272207, 'depth': 4, 'l2_leaf_reg': 13.70317538683857, 'random_strength': 0.1755724094154989, 'grow_policy': 'SymmetricTree', 'leaf_estimation_method': 'Gradient', 'leaf_estimation_iterations': 1, 'subsample': 0.6972589059086052}. Best is trial 1 with value: 0.6553038926072227.
[I 2026-04-15 14:57:32,636] Trial 2 finished with value: 0.6388479360883415 and parameters: {'bootstrap_type': 'Bernoulli', 'l

,feature,importance
56,eye__sac_length_AOI_aggregated_Pos_median,7.251115
53,eye__sac_length_AOI_aggregated_Pos_min,6.763889
63,eye__sac_speed_AOI_aggregated_Pos_median,5.728875
60,eye__sac_speed_AOI_aggregated_Pos_min,4.064787
62,eye__sac_speed_AOI_aggregated_Pos_mean,2.672050
178,eye__fix_duration_AOI_FD_Neg1_mean,2.356228
276,eye__fix_duration_AOI_aggregated_Pos_min,2.214647
61,eye__sac_speed_AOI_aggregated_Pos_max,2.099593
168,eye__fix_duration_mean,1.883088
54,eye__sac_length_AOI_aggregated_Pos_max,1.757331


,set,target,split,no_neutral,xgb_test,catboost_test,file
0,envelope,match_mismatch,binary,False,0.399740,0.410272,C:\Users\LEGION\Projects\CB_exepriment\dataset...
1,envelope,match_mismatch,binary,True,0.403197,0.476267,C:\Users\LEGION\Projects\CB_exepriment\dataset...
2,envelope,match_mismatch,multiclass,False,0.621856,0.621466,C:\Users\LEGION\Projects\CB_exepriment\dataset...
3,envelope,match_mismatch,multiclass,True,NaN,0.505935,C:\Users\LEGION\Projects\CB_exepriment\dataset...
4,envelope,match_mismatch_general,binary,False,0.398957,0.554178,C:\Users\LEGION\Projects\CB_exepriment\dataset...
5,envelope,match_mismatch_general,binary,True,0.410526,0.542756,C:\Users\LEGION\Projects\CB_exepriment\dataset...
6,envelope,match_mismatch_general,multiclass,False,0.806555,0.808673,C:\Users\LEGION\Projects\CB_exepriment\dataset...
7,envelope,match_mismatch_general,multiclass,True,NaN,0.820915,C:\Users\LEGION\Projects\CB_exepriment\dataset...


,set,target,split,no_neutral,xgb_test,catboost_test,file
0,envelope,match_mismatch,binary,False,0.399740,0.410272,C:\Users\LEGION\Projects\CB_exepriment\dataset...
1,envelope,match_mismatch,binary,True,0.403197,0.476267,C:\Users\LEGION\Projects\CB_exepriment\dataset...
2,envelope,match_mismatch,multiclass,False,0.621856,0.621466,C:\Users\LEGION\Projects\CB_exepriment\dataset...
3,envelope,match_mismatch,multiclass,True,NaN,0.505935,C:\Users\LEGION\Projects\CB_exepriment\dataset...
4,envelope,match_mismatch_general,binary,False,0.398957,0.554178,C:\Users\LEGION\Projects\CB_exepriment\dataset...
5,envelope,match_mismatch_general,binary,True,0.410526,0.542756,C:\Users\LEGION\Projects\CB_exepriment\dataset...
6,envelope,match_mismatch_general,multiclass,False,0.806555,0.808673,C:\Users\LEGION\Projects\CB_exepriment\dataset...
7,envelope,match_mismatch_general,multiclass,True,NaN,0.820915,C:\Users\LEGION\Projects\CB_exepriment\dataset...


In [8]:
text_freq_bands = _load_ftar_parquet(FTAR_FILES["freq_bands"])
print("freq_bands shape:", text_freq_bands.shape)
summary_freq_bands = run_ftar_set("freq_bands", text_freq_bands)
summary_freq_bands

freq_bands shape: (4442, 18362)
[freq_bands] ftar feature count after cleanup: 18361

Skipping existing experiment: exp__X_name=screen+freq_bands__target=match_mismatch__problem=binary__feat=all_features+freq_bands__ES__refitTEST
[freq_bands] rows missing after index match -> train: 1658, test: 160
[freq_bands] UMAP compression: train/test features 18361 -> 100


c:\Users\LEGION\miniconda3\envs\table_methods\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
[I 2026-04-15 15:30:20,557] A new study created in memory with name: binary_xgb



Running: exp__X_name=screen+freq_bands__target=match_mismatch__problem=binary__feat=all_features+freq_bands__ES__refitTEST__no_neutral
X_train: (3016, 570), X_test: (336, 570)


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-04-15 15:30:24,610] Trial 0 finished with value: 0.40517617025578806 and parameters: {'learning_rate': 0.04534239315199742, 'max_depth': 4, 'min_child_weight': 3.6133378418934727, 'subsample': 0.7172121707956832, 'colsample_bytree': 0.7642269046455532, 'gamma': 0.2930268354654828, 'reg_alpha': 0.0005749550436838274, 'reg_lambda': 2.9181778067053106}. Best is trial 0 with value: 0.40517617025578806.
[I 2026-04-15 15:30:29,598] Trial 1 finished with value: 0.40517617025578806 and parameters: {'learning_rate': 0.1707594054749505, 'max_depth': 8, 'min_child_weight': 1.8407677988385938, 'subsample': 0.9514458866935752, 'colsample_bytree': 0.7320289895353271, 'gamma': 0.030927525032881986, 'reg_alpha': 2.3384483013779853, 'reg_lambda': 0.0007548524119611296}. Best is trial 0 with value: 0.40517617025578806.
[I 2026-04-15 15:30:34,640] Trial 2 finished with value: 0.40517617025578806 and parameters: {'learning_rate': 0.0333616681035175, 'max_depth': 5, 'min_child_weight': 1.3687627858

...

  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-04-15 17:13:33,374] Trial 0 finished with value: 0.6317636845364394 and parameters: {'bootstrap_type': 'Bayesian', 'learning_rate': 0.07226675683786944, 'depth': 4, 'l2_leaf_reg': 1.8990530558141054, 'random_strength': 0.1465134177327414, 'grow_policy': 'SymmetricTree', 'leaf_estimation_method': 'Gradient', 'leaf_estimation_iterations': 5, 'bagging_temperature': 4.973275478387985}. Best is trial 0 with value: 0.6317636845364394.
[I 2026-04-15 17:13:44,028] Trial 1 finished with value: 0.6482380892967529 and parameters: {'bootstrap_type': 'Bernoulli', 'learning_rate': 0.025573683975272207, 'depth': 4, 'l2_leaf_reg': 13.70317538683857, 'random_strength': 0.1755724094154989, 'grow_policy': 'SymmetricTree', 'leaf_estimation_method': 'Gradient', 'leaf_estimation_iterations': 1, 'subsample': 0.6972589059086052}. Best is trial 1 with value: 0.6482380892967529.
[I 2026-04-15 17:13:57,507] Trial 2 finished with value: 0.6461341570193632 and parameters: {'bootstrap_type': 'Bernoulli', 'l

,feature,importance
56,eye__sac_length_AOI_aggregated_Pos_median,11.894400
58,eye__sac_length_AOI_aggregated_Pos_sum,11.135280
275,eye__fix_duration_AOI_aggregated_Pos_first,9.872559
53,eye__sac_length_AOI_aggregated_Pos_min,6.411800
63,eye__sac_speed_AOI_aggregated_Pos_median,5.032136
62,eye__sac_speed_AOI_aggregated_Pos_mean,3.246429
532,freq_bands__umap_062,3.245042
490,freq_bands__umap_020,3.215304
0,eye__sac_length_min,2.909855
166,eye__fix_duration_min,2.782568


,set,target,split,no_neutral,xgb_test,catboost_test,file
0,freq_bands,match_mismatch,binary,False,0.399740,0.416763,C:\Users\LEGION\Projects\CB_exepriment\dataset...
1,freq_bands,match_mismatch,binary,True,0.403197,0.443463,C:\Users\LEGION\Projects\CB_exepriment\dataset...
2,freq_bands,match_mismatch,multiclass,False,0.633477,0.628659,C:\Users\LEGION\Projects\CB_exepriment\dataset...
3,freq_bands,match_mismatch,multiclass,True,NaN,0.544316,C:\Users\LEGION\Projects\CB_exepriment\dataset...
4,freq_bands,match_mismatch_general,binary,False,0.484267,0.531461,C:\Users\LEGION\Projects\CB_exepriment\dataset...
5,freq_bands,match_mismatch_general,binary,True,0.410526,0.588046,C:\Users\LEGION\Projects\CB_exepriment\dataset...
6,freq_bands,match_mismatch_general,multiclass,False,0.808328,0.801248,C:\Users\LEGION\Projects\CB_exepriment\dataset...
7,freq_bands,match_mismatch_general,multiclass,True,NaN,0.708828,C:\Users\LEGION\Projects\CB_exepriment\dataset...


,set,target,split,no_neutral,xgb_test,catboost_test,file
0,freq_bands,match_mismatch,binary,False,0.399740,0.416763,C:\Users\LEGION\Projects\CB_exepriment\dataset...
1,freq_bands,match_mismatch,binary,True,0.403197,0.443463,C:\Users\LEGION\Projects\CB_exepriment\dataset...
2,freq_bands,match_mismatch,multiclass,False,0.633477,0.628659,C:\Users\LEGION\Projects\CB_exepriment\dataset...
3,freq_bands,match_mismatch,multiclass,True,NaN,0.544316,C:\Users\LEGION\Projects\CB_exepriment\dataset...
4,freq_bands,match_mismatch_general,binary,False,0.484267,0.531461,C:\Users\LEGION\Projects\CB_exepriment\dataset...
5,freq_bands,match_mismatch_general,binary,True,0.410526,0.588046,C:\Users\LEGION\Projects\CB_exepriment\dataset...
6,freq_bands,match_mismatch_general,multiclass,False,0.808328,0.801248,C:\Users\LEGION\Projects\CB_exepriment\dataset...
7,freq_bands,match_mismatch_general,multiclass,True,NaN,0.708828,C:\Users\LEGION\Projects\CB_exepriment\dataset...


In [9]:
text_PID = _load_ftar_parquet(FTAR_FILES["PID"])
print("PID shape:", text_PID.shape)
summary_PID = run_ftar_set("PID", text_PID)
summary_PID

PID shape: (4442, 611)
[PID] ftar feature count after cleanup: 610

Skipping existing experiment: exp__X_name=screen+PID__target=match_mismatch__problem=binary__feat=all_features+PID__ES__refitTEST


[I 2026-04-15 17:28:11,384] A new study created in memory with name: binary_xgb


[PID] rows missing after index match -> train: 1658, test: 160

Running: exp__X_name=screen+PID__target=match_mismatch__problem=binary__feat=all_features+PID__ES__refitTEST__no_neutral
X_train: (3016, 1080), X_test: (336, 1080)


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-04-15 17:28:20,528] Trial 0 finished with value: 0.40517617025578806 and parameters: {'learning_rate': 0.04534239315199742, 'max_depth': 4, 'min_child_weight': 3.6133378418934727, 'subsample': 0.7172121707956832, 'colsample_bytree': 0.7642269046455532, 'gamma': 0.2930268354654828, 'reg_alpha': 0.0005749550436838274, 'reg_lambda': 2.9181778067053106}. Best is trial 0 with value: 0.40517617025578806.
[I 2026-04-15 17:28:31,650] Trial 1 finished with value: 0.40517617025578806 and parameters: {'learning_rate': 0.1707594054749505, 'max_depth': 8, 'min_child_weight': 1.8407677988385938, 'subsample': 0.9514458866935752, 'colsample_bytree': 0.7320289895353271, 'gamma': 0.030927525032881986, 'reg_alpha': 2.3384483013779853, 'reg_lambda': 0.0007548524119611296}. Best is trial 0 with value: 0.40517617025578806.
[I 2026-04-15 17:28:40,763] Trial 2 finished with value: 0.40517617025578806 and parameters: {'learning_rate': 0.0333616681035175, 'max_depth': 5, 'min_child_weight': 1.3687627858

...

  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-04-15 21:31:48,782] Trial 0 finished with value: 0.62227135023457 and parameters: {'bootstrap_type': 'Bayesian', 'learning_rate': 0.07226675683786944, 'depth': 4, 'l2_leaf_reg': 1.8990530558141054, 'random_strength': 0.1465134177327414, 'grow_policy': 'SymmetricTree', 'leaf_estimation_method': 'Gradient', 'leaf_estimation_iterations': 5, 'bagging_temperature': 4.973275478387985}. Best is trial 0 with value: 0.62227135023457.
[I 2026-04-15 21:32:13,421] Trial 1 finished with value: 0.6486174000448082 and parameters: {'bootstrap_type': 'Bernoulli', 'learning_rate': 0.025573683975272207, 'depth': 4, 'l2_leaf_reg': 13.70317538683857, 'random_strength': 0.1755724094154989, 'grow_policy': 'SymmetricTree', 'leaf_estimation_method': 'Gradient', 'leaf_estimation_iterations': 1, 'subsample': 0.6972589059086052}. Best is trial 1 with value: 0.6486174000448082.
[I 2026-04-15 21:32:45,222] Trial 2 finished with value: 0.6423570812916353 and parameters: {'bootstrap_type': 'Bernoulli', 'learn

,feature,importance
56,eye__sac_length_AOI_aggregated_Pos_median,11.074327
63,eye__sac_speed_AOI_aggregated_Pos_median,7.075666
53,eye__sac_length_AOI_aggregated_Pos_min,7.040510
62,eye__sac_speed_AOI_aggregated_Pos_mean,6.269328
60,eye__sac_speed_AOI_aggregated_Pos_min,4.125146
544,PID__PID_CP4_second_deriv_mean,2.936185
178,eye__fix_duration_AOI_FD_Neg1_mean,2.328713
284,eye__fix_duration_AOI_aggregated_Pos_count,2.043635
166,eye__fix_duration_min,1.941292
176,eye__fix_duration_AOI_FD_Neg1_min,1.847281


,set,target,split,no_neutral,xgb_test,catboost_test,file
0,PID,match_mismatch,binary,False,0.399740,0.402184,C:\Users\LEGION\Projects\CB_exepriment\dataset...
1,PID,match_mismatch,binary,True,0.403197,0.404151,C:\Users\LEGION\Projects\CB_exepriment\dataset...
2,PID,match_mismatch,multiclass,False,0.623863,0.614744,C:\Users\LEGION\Projects\CB_exepriment\dataset...
3,PID,match_mismatch,multiclass,True,NaN,0.540870,C:\Users\LEGION\Projects\CB_exepriment\dataset...
4,PID,match_mismatch_general,binary,False,0.404235,0.567179,C:\Users\LEGION\Projects\CB_exepriment\dataset...
5,PID,match_mismatch_general,binary,True,0.500321,0.603175,C:\Users\LEGION\Projects\CB_exepriment\dataset...
6,PID,match_mismatch_general,multiclass,False,0.802558,0.814880,C:\Users\LEGION\Projects\CB_exepriment\dataset...
7,PID,match_mismatch_general,multiclass,True,NaN,0.745125,C:\Users\LEGION\Projects\CB_exepriment\dataset...


,set,target,split,no_neutral,xgb_test,catboost_test,file
0,PID,match_mismatch,binary,False,0.399740,0.402184,C:\Users\LEGION\Projects\CB_exepriment\dataset...
1,PID,match_mismatch,binary,True,0.403197,0.404151,C:\Users\LEGION\Projects\CB_exepriment\dataset...
2,PID,match_mismatch,multiclass,False,0.623863,0.614744,C:\Users\LEGION\Projects\CB_exepriment\dataset...
3,PID,match_mismatch,multiclass,True,NaN,0.540870,C:\Users\LEGION\Projects\CB_exepriment\dataset...
4,PID,match_mismatch_general,binary,False,0.404235,0.567179,C:\Users\LEGION\Projects\CB_exepriment\dataset...
5,PID,match_mismatch_general,binary,True,0.500321,0.603175,C:\Users\LEGION\Projects\CB_exepriment\dataset...
6,PID,match_mismatch_general,multiclass,False,0.802558,0.814880,C:\Users\LEGION\Projects\CB_exepriment\dataset...
7,PID,match_mismatch_general,multiclass,True,NaN,0.745125,C:\Users\LEGION\Projects\CB_exepriment\dataset...


In [10]:
text_stat = _load_ftar_parquet(FTAR_FILES["stat"])
print("stat shape:", text_stat.shape)
summary_stat = run_ftar_set("stat", text_stat)
summary_stat

stat shape: (4442, 8236)
[stat] ftar feature count after cleanup: 8235

Skipping existing experiment: exp__X_name=screen+stat__target=match_mismatch__problem=binary__feat=all_features+stat__ES__refitTEST
[stat] rows missing after index match -> train: 1658, test: 160
[stat] UMAP compression: train/test features 8235 -> 100


c:\Users\LEGION\miniconda3\envs\table_methods\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
[I 2026-04-15 22:04:21,194] A new study created in memory with name: binary_xgb



Running: exp__X_name=screen+stat__target=match_mismatch__problem=binary__feat=all_features+stat__ES__refitTEST__no_neutral
X_train: (3016, 570), X_test: (336, 570)


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-04-15 22:04:25,232] Trial 0 finished with value: 0.40517617025578806 and parameters: {'learning_rate': 0.04534239315199742, 'max_depth': 4, 'min_child_weight': 3.6133378418934727, 'subsample': 0.7172121707956832, 'colsample_bytree': 0.7642269046455532, 'gamma': 0.2930268354654828, 'reg_alpha': 0.0005749550436838274, 'reg_lambda': 2.9181778067053106}. Best is trial 0 with value: 0.40517617025578806.
[I 2026-04-15 22:04:30,142] Trial 1 finished with value: 0.40352843125103893 and parameters: {'learning_rate': 0.1707594054749505, 'max_depth': 8, 'min_child_weight': 1.8407677988385938, 'subsample': 0.9514458866935752, 'colsample_bytree': 0.7320289895353271, 'gamma': 0.030927525032881986, 'reg_alpha': 2.3384483013779853, 'reg_lambda': 0.0007548524119611296}. Best is trial 0 with value: 0.40517617025578806.
[I 2026-04-15 22:04:34,946] Trial 2 finished with value: 0.40517617025578806 and parameters: {'learning_rate': 0.0333616681035175, 'max_depth': 5, 'min_child_weight': 1.3687627858

...

  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-04-15 23:27:44,262] Trial 0 finished with value: 0.6294443427315789 and parameters: {'bootstrap_type': 'Bayesian', 'learning_rate': 0.07226675683786944, 'depth': 4, 'l2_leaf_reg': 1.8990530558141054, 'random_strength': 0.1465134177327414, 'grow_policy': 'SymmetricTree', 'leaf_estimation_method': 'Gradient', 'leaf_estimation_iterations': 5, 'bagging_temperature': 4.973275478387985}. Best is trial 0 with value: 0.6294443427315789.
[I 2026-04-15 23:27:53,972] Trial 1 finished with value: 0.6514023595772884 and parameters: {'bootstrap_type': 'Bernoulli', 'learning_rate': 0.025573683975272207, 'depth': 4, 'l2_leaf_reg': 13.70317538683857, 'random_strength': 0.1755724094154989, 'grow_policy': 'SymmetricTree', 'leaf_estimation_method': 'Gradient', 'leaf_estimation_iterations': 1, 'subsample': 0.6972589059086052}. Best is trial 1 with value: 0.6514023595772884.
[I 2026-04-15 23:28:08,976] Trial 2 finished with value: 0.6463619033266641 and parameters: {'bootstrap_type': 'Bernoulli', 'l

,feature,importance
56,eye__sac_length_AOI_aggregated_Pos_median,13.210942
275,eye__fix_duration_AOI_aggregated_Pos_first,9.439212
58,eye__sac_length_AOI_aggregated_Pos_sum,7.775808
53,eye__sac_length_AOI_aggregated_Pos_min,6.267648
63,eye__sac_speed_AOI_aggregated_Pos_median,4.806906
278,eye__fix_duration_AOI_aggregated_Pos_mean,3.371740
542,stat__umap_072,3.205570
491,stat__umap_021,2.787353
276,eye__fix_duration_AOI_aggregated_Pos_min,2.603435
508,stat__umap_038,2.281582


,set,target,split,no_neutral,xgb_test,catboost_test,file
0,stat,match_mismatch,binary,False,0.399740,0.436073,C:\Users\LEGION\Projects\CB_exepriment\dataset...
1,stat,match_mismatch,binary,True,0.403197,0.449908,C:\Users\LEGION\Projects\CB_exepriment\dataset...
2,stat,match_mismatch,multiclass,False,0.626858,0.618859,C:\Users\LEGION\Projects\CB_exepriment\dataset...
3,stat,match_mismatch,multiclass,True,NaN,0.568418,C:\Users\LEGION\Projects\CB_exepriment\dataset...
4,stat,match_mismatch_general,binary,False,0.439737,0.554900,C:\Users\LEGION\Projects\CB_exepriment\dataset...
5,stat,match_mismatch_general,binary,True,0.410526,0.553716,C:\Users\LEGION\Projects\CB_exepriment\dataset...
6,stat,match_mismatch_general,multiclass,False,0.801435,0.817319,C:\Users\LEGION\Projects\CB_exepriment\dataset...
7,stat,match_mismatch_general,multiclass,True,NaN,0.800452,C:\Users\LEGION\Projects\CB_exepriment\dataset...


,set,target,split,no_neutral,xgb_test,catboost_test,file
0,stat,match_mismatch,binary,False,0.399740,0.436073,C:\Users\LEGION\Projects\CB_exepriment\dataset...
1,stat,match_mismatch,binary,True,0.403197,0.449908,C:\Users\LEGION\Projects\CB_exepriment\dataset...
2,stat,match_mismatch,multiclass,False,0.626858,0.618859,C:\Users\LEGION\Projects\CB_exepriment\dataset...
3,stat,match_mismatch,multiclass,True,NaN,0.568418,C:\Users\LEGION\Projects\CB_exepriment\dataset...
4,stat,match_mismatch_general,binary,False,0.439737,0.554900,C:\Users\LEGION\Projects\CB_exepriment\dataset...
5,stat,match_mismatch_general,binary,True,0.410526,0.553716,C:\Users\LEGION\Projects\CB_exepriment\dataset...
6,stat,match_mismatch_general,multiclass,False,0.801435,0.817319,C:\Users\LEGION\Projects\CB_exepriment\dataset...
7,stat,match_mismatch_general,multiclass,True,NaN,0.800452,C:\Users\LEGION\Projects\CB_exepriment\dataset...


In [11]:
all_summaries = [
    summary_corr,
    summary_cov_freq,
    summary_envelope,
    summary_freq_bands,
    summary_PID,
    summary_stat,
]
all_results = pd.concat(all_summaries, ignore_index=True)
all_results

,set,target,split,no_neutral,xgb_test,catboost_test,file
0,corr,match_mismatch,binary,False,0.398399,0.414870,C:\Users\LEGION\Projects\CB_exepriment\dataset...
1,corr,match_mismatch,binary,True,0.403197,0.443771,C:\Users\LEGION\Projects\CB_exepriment\dataset...
2,corr,match_mismatch,multiclass,False,0.596315,0.608868,C:\Users\LEGION\Projects\CB_exepriment\dataset...
3,corr,match_mismatch,multiclass,True,NaN,0.541407,C:\Users\LEGION\Projects\CB_exepriment\dataset...
4,corr,match_mismatch_general,binary,False,0.426946,0.529915,C:\Users\LEGION\Projects\CB_exepriment\dataset...
5,corr,match_mismatch_general,binary,True,0.440106,0.486745,C:\Users\LEGION\Projects\CB_exepriment\dataset...
6,corr,match_mismatch_general,multiclass,False,0.799472,0.808972,C:\Users\LEGION\Projects\CB_exepriment\dataset...
7,corr,match_mismatch_general,multiclass,True,NaN,0.773737,C:\Users\LEGION\Projects\CB_exepriment\dataset...
8,cov_freq,match_mismatch,binary,False,0.399740,0.403406,C:\Users\LEGION\Projects\CB_exepriment\dataset...
9,cov_freq,match_mismatch,binary,True,0.403197,0.505965,C:\Users\LEGION\Projects\CB_exepriment\dataset...


In [12]:
summary_path = OUT_DIR / "summary_eye_ftar.csv"
all_results.to_csv(summary_path, index=False)
print("Saved summary:", summary_path)

all_results.groupby(["set", "target", "split"], as_index=False)[["xgb_test", "catboost_test"]].mean()

Saved summary: C:\Users\LEGION\Projects\CB_exepriment\dataset_v2\optuna_results_eye_ftar\summary_eye_ftar.csv


,set,target,split,xgb_test,catboost_test
0,PID,match_mismatch,binary,0.401468,0.403168
1,PID,match_mismatch,multiclass,0.623863,0.577807
2,PID,match_mismatch_general,binary,0.452278,0.585177
3,PID,match_mismatch_general,multiclass,0.802558,0.780003
4,corr,match_mismatch,binary,0.400798,0.429320
5,corr,match_mismatch,multiclass,0.596315,0.575137
6,corr,match_mismatch_general,binary,0.433526,0.508330
7,corr,match_mismatch_general,multiclass,0.799472,0.791354
8,cov_freq,match_mismatch,binary,0.401468,0.454685
9,cov_freq,match_mismatch,multiclass,0.621595,0.586633
